### Procesador de Direcciones Independiente (Ejecutable sin iniciar sesión)

### 1. Funciones de Normalización de Direcciones y Configuración

In [ ]:
import re
import unicodedata
import pandas as pd
import io
import ipywidgets as widgets
from IPython.display import display

def normalizar_caracteres_especiales(texto):
    if not isinstance(texto, str):
        return ""

    # Protección especial para conservar la Ñ y ñ antes de remover diacríticos
    texto_protegido = texto.replace('Ñ', '[[MAY_N_TILDE]]').replace('ñ', '[[MIN_N_TILDE]]')

    # Reemplazar caracteres residuales comunes de codificación
    texto_protegido = re.sub(r'[Ââ]', lambda m: 'A' if m.group().isupper() else 'a', texto_protegido)

    # Descomponer caracteres Unicode para separar las tildes/diacríticos de su letra base
    nfkd_form = unicodedata.normalize('NFKD', texto_protegido)

    # Filtrar marcas de combinación (diacríticos), manteniendo el texto base
    texto_sin_diacriticos = "".join([c for c in nfkd_form if not unicodedata.combining(c)])

    # Restaurar la Ñ y ñ originales
    texto_restaurado = texto_sin_diacriticos.replace('[[MAY_N_TILDE]]', 'Ñ').replace('[[MIN_N_TILDE]]', 'ñ')

    return texto_restaurado

def normalizar_direccion(direccion):
    if pd.isna(direccion) or not str(direccion).strip():
        return ""

    # Asegurar que el valor se trate como texto para no perder ceros a la izquierda ni formatearlo como decimal
    texto = str(direccion)

    # 1. Tratamiento de espacios (reemplazar tabulaciones, NBSP y cualquier espacio Unicode por espacio normal ASCII)
    # \s en expresiones regulares con bandera Unicode detecta todos los espacios en blanco
    texto_espacios = re.sub(r'\s+', ' ', texto, flags=re.UNICODE)

    # 2. Normalización de letras con caracteres especiales (tildes, diacríticos, codificaciones extrañas)
    texto_normalizado = normalizar_caracteres_especiales(texto_espacios)

    # 3. Filtrar únicamente los caracteres permitidos:
    # Letras (A-Za-z), Ñ, ñ, Números (0-9), Espacios, '#' y '-'
    patron_permitidos = re.compile(r'[^A-Za-zÑñ0-9 #\-]')
    texto_filtrado = patron_permitidos.sub('', texto_normalizado)

    # 4. Segunda pasada de control de espacios para asegurar consistencia
    texto_final = re.sub(r' {2,}', ' ', texto_filtrado) # Reducir múltiples espacios a uno solo
    texto_final = texto_final.strip() # Quitar espacios al inicio y al final

    return texto_final

### 2. Procesamiento, Carga y Guardado Automático de Archivos Excel

In [ ]:
# Inicializar el cargador interactivo de archivos
uploader = widgets.FileUpload(accept='.xlsx', multiple=False)
print("Por favor, selecciona el archivo Excel (.xlsx) para procesar:")
display(uploader)

def procesar_archivo_excel(change):
    if not uploader.value:
        return

    # Obtener el nombre y contenido del archivo cargado desde la interfaz interactiva
    info_archivo = list(uploader.value.values())[0]
    nombre_archivo_original = info_archivo['metadata']['name']
    contenido_archivo = info_archivo['content']

    print(f"\nLeyendo y procesando archivo: {nombre_archivo_original}...")

    # Leer el archivo con pandas utilizando openpyxl.
    try:
        df = pd.read_excel(io.BytesIO(contenido_archivo), dtype=str, engine='openpyxl')
    except Exception as e:
        print(f"Error al leer el archivo Excel: {e}")
        return

    # Validación de la columna requerida
    columna_objetivo = 'Direccion'
    if columna_objetivo not in df.columns:
        print(f"[ERROR] No se encontró la columna exacta '{columna_objetivo}' en el archivo Excel.")
        print(f"Columnas detectadas: {list(df.columns)}")
        return

    # Aplicar la normalización fila por fila
    df['Direccion ajustada'] = df[columna_objetivo].apply(normalizar_direccion)

    # Calcular estadísticas de procesamiento
    total_registros = len(df)

    # Manejar nulos en la columna original para el conteo estadístico
    direcciones_vacias = df[columna_objetivo].apply(lambda x: pd.isna(x) or str(x).strip() == "").sum()

    # Determinar cuáles cambiaron
    mismas_direcciones = 0
    modificadas_direcciones = 0

    for idx, fila in df.iterrows():
        orig = str(fila[columna_objetivo]).strip() if not pd.isna(fila[columna_objetivo]) else ""
        ajust = fila['Direccion ajustada']
        if orig == "" and ajust == "":
            continue
        if orig == ajust:
            mismas_direcciones += 1
        else:
            modificadas_direcciones += 1

    # Preparar el archivo final de salida
    nombre_archivo_generado = nombre_archivo_original.replace(".xlsx", "_ajustado.xlsx")
    if nombre_archivo_generado == nombre_archivo_original:
        nombre_archivo_generado = "direcciones_ajustadas.xlsx"

    # Guardar en el almacenamiento virtual de Binder
    df.to_excel(nombre_archivo_generado, index=False, engine='openpyxl')

    # Mostrar Estadísticas de procesamiento
    print("\n" + "="*45)
    print("          ESTADÍSTICAS DEL PROCESO")
    print("="*45)
    print(f"Nombre del archivo original:  {nombre_archivo_original}")
    print(f"Cantidad total de registros:  {total_registros}")
    print(f"Direcciones modificadas:      {modificadas_direcciones}")
    print(f"Direcciones sin cambios:      {mismas_direcciones}")
    print(f"Direcciones vacías o nulas:   {direcciones_vacias}")
    print(f"Archivo guardado con éxito:   {nombre_archivo_generado}")
    print("="*45)
    print("\n[INFO] El archivo ya está listo. Para descargarlo, ve al panel izquierdo de Binder, haz clic derecho sobre el archivo modificado y selecciona 'Download'.\n")

# Observar cambios en el uploader para ejecutar automáticamente
uploader.observe(procesar_archivo_excel, names='value')